# Primer parcial: Random Forest (notebook independiente)
Reproduce, sin el análisis exploratorio ni el árbol, lo necesario de `primer_parcial.ipynb` para entrenar el Random Forest: carga de datos, limpieza de duplicados, pipeline de preprocesamiento, función de ganancia y separación train/test. Todas esas celdas son copia textual del notebook original.

**Requisitos:** Python 3.10+, `scikit-learn >= 1.4` (por los NaN en `precio_usd`), `pandas`, `numpy`, `matplotlib`, `seaborn` y conexión a internet (los CSV se leen de GitHub).
Instalar con: `pip install -U scikit-learn pandas numpy matplotlib seaborn`

**Tiempo:** la búsqueda de hiperparámetros entrena 100 bosques de 300 árboles. Usa todos los núcleos (`n_jobs=-1`): en una PC de 4 núcleos tarda del orden de 10-20 minutos.

**Salida:** `casas_entrega_random_forest.csv` en la carpeta del notebook.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
from IPython.display import display

# Carga de datos

In [ ]:
df_casas_entrenamiento = pd.read_csv('https://raw.githubusercontent.com/MartinDido100/ciencia-de-datos/refs/heads/main/primer_parcial/datasets/casas_entrenamiento.csv')
df_casas_entrenamiento.head(5)

In [ ]:
df_casas_predecir = pd.read_csv('https://raw.githubusercontent.com/MartinDido100/ciencia-de-datos/refs/heads/main/primer_parcial/datasets/casas_predecir.csv')
df_casas_predecir.head(5)

# Limpieza de duplicados
Misma limpieza que en `primer_parcial.ipynb` (ver ahí la justificación de cada paso).

## Limpieza de duplicados (1/3): casas con etiqueta contradictoria (16 grupos, 33 filas)

**Por qué:** hay propiedades idénticas en todas sus columnas (salvo `id`) que en una fila figuran como `paga` y en otra como `no paga`. Con los mismos datos de entrada, el modelo recibe dos respuestas opuestas: es ruido que no puede aprender. Como no hay forma de saber cuál etiqueta es la correcta, se eliminan **todas** las filas de esos grupos, sin elegir una. Este paso va antes de quitar los duplicados: si se deduplicara primero, cada grupo quedaría en una sola fila con una etiqueta elegida al azar y la contradicción ya no se vería.

**Qué hace:** calcula una "huella" de cada fila a partir de todas las columnas menos `id` y `paga_comision` (filas idénticas tienen la misma huella). Cuenta cuántas etiquetas distintas tiene cada huella y elimina del entrenamiento las filas cuya huella tiene más de una.

In [ ]:
# Columnas que describen la casa: todas menos id y el target
# (funciona tanto si id es columna como si ya es el índice)
features = [c for c in df_casas_entrenamiento.columns if c not in ('id', 'paga_comision')]

# Huella de cada fila: filas idénticas en 'features' tienen el mismo número (los NaN también se comparan bien)
huella_tr = pd.util.hash_pandas_object(df_casas_entrenamiento[features], index=False)

# Para cada fila, cuántas etiquetas distintas tiene su grupo de casas idénticas
etiquetas_en_grupo = df_casas_entrenamiento.groupby(huella_tr)['paga_comision'].transform('nunique')
contradictorias = etiquetas_en_grupo > 1

filas_antes = len(df_casas_entrenamiento)
print("Grupos con etiqueta contradictoria:", huella_tr[contradictorias].nunique())
print("Filas a eliminar (todas las de esos grupos):", contradictorias.sum())

df_casas_entrenamiento = df_casas_entrenamiento[~contradictorias]
print(f"Entrenamiento: {filas_antes} -> {len(df_casas_entrenamiento)} filas")

# Verificación: ya no queda ninguna casa con dos etiquetas distintas
huella_tr = pd.util.hash_pandas_object(df_casas_entrenamiento[features], index=False)
assert (df_casas_entrenamiento.groupby(huella_tr)['paga_comision'].nunique() == 1).all()

## Limpieza de duplicados (2/3): duplicados dentro del entrenamiento

**Por qué:** una misma casa publicada varias veces con distinto `id` pesa doble en el entrenamiento, y al separar train/test una copia puede quedar de cada lado. En ese caso el modelo "reconoce" la casa en vez de predecirla, sobreajusta y la ganancia estimada en test sale más alta de lo que va a ser con casas nuevas. En `casas_predecir` no se eliminan: hay que entregar una predicción para cada `id`.

**Qué hace:** elimina las filas del entrenamiento que son idénticas a otra en todas las columnas menos `id` (la etiqueta ya coincide, porque las contradictorias se quitaron en el paso anterior). Conserva la primera aparición de cada casa.

In [ ]:
# Duplicados de la misma casa con distinto id (se ignora id; la etiqueta ya es consistente)
features = [c for c in df_casas_entrenamiento.columns if c not in ('id', 'paga_comision')]

filas_antes = len(df_casas_entrenamiento)
print("Duplicados a eliminar:", df_casas_entrenamiento.duplicated(subset=features).sum())

# keep='first': de cada grupo de casas idénticas queda la primera fila
df_casas_entrenamiento = df_casas_entrenamiento.drop_duplicates(subset=features, keep='first')
print(f"Entrenamiento: {filas_antes} -> {len(df_casas_entrenamiento)} filas")

# Verificación: no quedan casas repetidas en el entrenamiento
assert not df_casas_entrenamiento.duplicated(subset=features).any()

## Limpieza de duplicados (3/3): casas de entrenamiento que también están en predecir

**Por qué:** hay casas de `casas_predecir` idénticas a casas del entrenamiento. `casas_predecir` es el dataset de producción y se predice completo, así que de ahí no se puede sacar nada; las coincidencias se quitan del entrenamiento. Así los dos conjuntos quedan disjuntos: el modelo se entrena y se valida solo con casas que no va a tener que predecir, y su rendimiento estimado refleja lo que va a pasar con casas nuevas, no con casas que ya vio.

**Qué hace:** calcula la huella (todas las columnas menos `id` y `paga_comision`) de cada fila de los dos archivos y elimina del entrenamiento las filas cuya huella aparece en `casas_predecir`. `casas_predecir` no se modifica.

In [ ]:
# Casas del entrenamiento idénticas a alguna de predecir (sin contar id ni target)
features = [c for c in df_casas_entrenamiento.columns if c not in ('id', 'paga_comision')]

huella_tr = pd.util.hash_pandas_object(df_casas_entrenamiento[features], index=False)
huella_pr = pd.util.hash_pandas_object(df_casas_predecir[features], index=False)
en_predecir = huella_tr.isin(huella_pr)

filas_antes, filas_predecir = len(df_casas_entrenamiento), len(df_casas_predecir)
print("Filas de entrenamiento que también están en predecir:", en_predecir.sum())
print("Filas de predecir afectadas (se mantienen):", huella_pr.isin(huella_tr).sum())

df_casas_entrenamiento = df_casas_entrenamiento[~en_predecir]
print(f"Entrenamiento: {filas_antes} -> {len(df_casas_entrenamiento)} filas")

# Verificaciones: los conjuntos quedaron disjuntos y predecir no perdió filas
huella_tr = pd.util.hash_pandas_object(df_casas_entrenamiento[features], index=False)
assert not huella_tr.isin(huella_pr).any()
assert len(df_casas_predecir) == filas_predecir

# Preprocesamiento (pipeline)
Aplica las decisiones del análisis exploratorio. Al final se exportan los dos datasets procesados para inspeccionarlos.

In [ ]:
import re
import unicodedata
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, TargetEncoder

# ============================================================================
# 0) FUERA DEL PIPELINE: id como índice y separación de X / y.
#    El target nunca va dentro de las predictoras (error común), y el Pipeline
#    estándar no puede eliminar filas, por eso la limpieza de duplicados quedó antes.
# ============================================================================
def id_como_indice(df):
    # id no describe la casa, pero lo necesitamos para el entregable
    return df.set_index('id') if 'id' in df.columns else df

entrenamiento = id_como_indice(df_casas_entrenamiento)
y = (entrenamiento['paga_comision'] == 'paga').astype(int)                    # 1 = paga, 0 = no paga
X = entrenamiento.drop(columns=['paga_comision'])
X_predecir = id_como_indice(df_casas_predecir).drop(columns=['paga_comision'])  # en predecir es relleno


# ============================================================================
# 1) PASOS "STATELESS" -> FunctionTransformer
#    Reglas fijas que no aprenden nada de los datos: dan lo mismo en train,
#    test y predecir, así que no pueden generar fuga.
# ============================================================================

# ad_type: un solo valor | l6: 100% vacía | created_on: idéntica a start_date
# lat/lon, l4, l5, price_period: decisión del análisis
# start_date/end_date: casi no cambian la tasa de pago (quedan fuera del modelo base)
COLUMNAS_A_ELIMINAR = ['ad_type', 'l6', 'created_on', 'lat', 'lon', 'l4', 'l5',
                       'price_period', 'start_date', 'end_date']

def eliminar_columnas(X):
    return X.drop(columns=COLUMNAS_A_ELIMINAR)


def invalidos_a_nan(X):
    # Valores imposibles -> NaN (no se borran filas: en predecir hay que predecir todas)
    X = X.copy()
    # bedrooms: 0 es un faltante disfrazado (casas de US$ 550.000 con 0 dormitorios); negativos y > 15 son imposibles
    X.loc[(X['bedrooms'] <= 0) | (X['bedrooms'] > 15), 'bedrooms'] = np.nan
    # bathrooms: más de 8 baños se considera error de carga
    X.loc[X['bathrooms'] > 8, 'bathrooms'] = np.nan
    # superficies: 0, negativas o menores a 20 m² no son posibles para una casa
    X.loc[X['surface_total'] < 20, 'surface_total'] = np.nan
    X.loc[X['surface_covered'] < 20, 'surface_covered'] = np.nan
    # cubierta > total es inconsistente: se descarta la total (la cubierta tiene menos errores)
    X.loc[X['surface_covered'] > X['surface_total'], 'surface_total'] = np.nan
    return X


# Marcas de moneda en dólares que aceptamos en el título. '$' solo NO: es ambiguo (pesos o dólares)
PATRON_PRECIO = re.compile(r'(?:u\$s|u\$d|usd|us\$|u\$)\s*(\d[\d\.,]*)')

def normalizar_texto(texto):
    # Minúsculas y sin tildes, para que 'Dúplex' y 'DUPLEX' se detecten igual
    texto = unicodedata.normalize('NFKD', str(texto).lower())
    return ''.join(c for c in texto if not unicodedata.combining(c))

def extraer_precio_usd(texto):
    # Busca 'U$S 190.000', 'USD 580.000', 'U$425.000', etc. Devuelve el número, o NaN si no hay precio
    m = PATRON_PRECIO.search(texto)
    if not m:
        return np.nan
    numero = re.sub(r'[\.,]\d{1,2}$', '', m.group(1).rstrip('.,'))  # saca decimales tipo ',00'
    numero = re.sub(r'[\.,]', '', numero)                            # saca separadores de miles
    if not numero:
        return np.nan
    valor = float(numero)
    # Fuera de un rango razonable es un número cortado ('U$D 550.') o mal escrito: no se usa
    return valor if 10_000 <= valor <= 50_000_000 else np.nan


def crear_features(X):
    X = X.copy()

    # l3 (ciudad): el nulo pasa a ser una categoría propia, porque tiene su propia tasa de pago
    X['l3'] = X['l3'].fillna('Sin dato')

    # Indicadores de nulo: se crean ANTES de imputar, porque que falte el dato informa sobre el target
    for col in ['rooms', 'bedrooms', 'bathrooms', 'surface_total', 'surface_covered']:
        X[f'{col}_faltante'] = X[col].isna().astype(int)

    # m2_descubiertos: patio/jardín (total - cubierta). Con m² reales, antes del log y de imputar
    X['m2_descubiertos'] = X['surface_total'] - X['surface_covered']

    # Imputación por regla de dominio (en Argentina, ambientes = dormitorios + 1)
    rooms_orig, bedrooms_orig = X['rooms'].copy(), X['bedrooms'].copy()
    X['rooms'] = rooms_orig.fillna(bedrooms_orig + 1)
    X['bedrooms'] = bedrooms_orig.fillna((rooms_orig - 1).where(rooms_orig >= 2))
    # surface_total sin dato pero con cubierta: la cubierta es un mínimo seguro de la total
    X['surface_total'] = X['surface_total'].fillna(X['surface_covered'])

    # Superficies en log: tienen cola larga (hasta 200.000 m²) y el log la comprime sin borrar datos
    X['surface_total'] = np.log1p(X['surface_total'])
    X['surface_covered'] = np.log1p(X['surface_covered'])

    # --- Variables nuevas a partir de title ---
    titulo = X['title'].fillna('').map(normalizar_texto)
    # Alquileres: solo nos importan las ventas, así que no se guarda como columna.
    # Se detectan únicamente para NO tomar su precio (es de alquiler, no de venta)
    es_alquiler = titulo.str.contains(r'\b(?:alquiler|alquilo|temporada)\b', regex=True)
    # precio_usd: la variable más fuerte (con precio > 500k paga el 99,9%). NaN si el título no lo trae
    X['precio_usd'] = titulo.map(extraer_precio_usd).where(~es_alquiler)
    X['tiene_precio'] = X['precio_usd'].notna().astype(int)
    # Tipo de propiedad: el dataset no lo tiene como columna y la tasa de pago varía mucho según el tipo
    # (es_lote se descartó: casi siempre el lote viene con una casa, así que la palabra confunde y no aporta)
    X['es_duplex'] = titulo.str.contains(r'\bduplex\b', regex=True).astype(int)
    X['es_ph'] = titulo.str.contains(r'\bph\b', regex=True).astype(int)
    X['es_depto'] = titulo.str.contains(r'\b(?:depto|dpto|departamento)s?\b', regex=True).astype(int)
    X['es_chalet'] = titulo.str.contains(r'\bchalet\b', regex=True).astype(int)
    # Palabras de lujo o de estado: suben (piscina, country, barrio cerrado) o bajan (a reciclar) la tasa
    X['tiene_piscina'] = titulo.str.contains(r'\b(?:piscina|pileta)\b', regex=True).astype(int)
    X['es_country'] = titulo.str.contains(r'\bcountry\b', regex=True).astype(int)
    X['barrio_cerrado'] = titulo.str.contains(r'\bbarrio cerrado\b', regex=True).astype(int)
    X['a_reciclar'] = titulo.str.contains(r'reciclar', regex=True).astype(int)

    # El texto original ya no aporta: todo lo útil quedó en las columnas nuevas
    return X.drop(columns=['title'])


# ============================================================================
# 2) PASO CON MEMORIA -> CustomTransformer
#    El tope de surface_covered se APRENDE en fit (solo con train) y se APLICA
#    en transform (train, test y predecir). Un FunctionTransformer no sirve acá
#    porque no tiene memoria.
# ============================================================================
class RecortarMaximo(BaseEstimator, TransformerMixin):
    # Recorta valores absurdos (ej: 235.000 m² cubiertos) al percentil indicado, calculado en fit
    def __init__(self, columnas=None, percentil=0.999):
        self.columnas = columnas
        self.percentil = percentil

    def fit(self, X, y=None):
        self.topes_ = {c: X[c].quantile(self.percentil) for c in self.columnas}
        self.feature_names_in_ = np.asarray(X.columns)
        return self

    def get_feature_names_out(self, input_features=None):
        return self.feature_names_in_

    def transform(self, X):
        Xc = X.copy()  # copia: no modificamos el DataFrame que nos pasan
        for c, tope in self.topes_.items():
            Xc[c] = Xc[c].clip(upper=tope)
        return Xc


# ============================================================================
# 3) ColumnTransformer: transformaciones por columna que APRENDEN de los datos
#    (categorías, tasas por ciudad, medianas). set_output pandas conserva los nombres.
# ============================================================================
cols_numericas = ['rooms', 'bedrooms', 'bathrooms', 'surface_total', 'surface_covered',
                  'm2_descubiertos']
cols_binarias = ['rooms_faltante', 'bedrooms_faltante', 'bathrooms_faltante',
                 'surface_total_faltante', 'surface_covered_faltante',
                 'tiene_precio', 'es_duplex', 'es_ph', 'es_depto', 'es_chalet',
                 'tiene_piscina', 'es_country', 'barrio_cerrado', 'a_reciclar']

ct_encoding = ColumnTransformer([
    # l1 (país, 3 valores): one-hot; un país nuevo en predecir queda en ceros en vez de dar error
    ('l1', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['l1']),
    # l2 (región, 38 valores): one-hot agrupando las regiones con < 50 casas, cuya tasa es puro ruido
    ('l2', OneHotEncoder(min_frequency=50, handle_unknown='infrequent_if_exist', sparse_output=False), ['l2']),
    # l3 (ciudad, 560 valores): target encoding = tasa de pago de la ciudad, con validación cruzada interna
    ('l3', TargetEncoder(target_type='binary', random_state=42), ['l3']),
    # numéricas: lo que sigue faltando se imputa con la mediana de train (robusta a outliers)
    ('num', SimpleImputer(strategy='median'), cols_numericas),
    # precio_usd: NO se imputa, queda NaN cuando el título no trae precio. Imputarlo con la
    # mediana inventaría un precio en el 83% de las filas; el NaN dice "no se sabe"
    ('precio', 'passthrough', ['precio_usd']),
    # binarias: ya están listas (0/1), pasan sin cambios
    ('bin', 'passthrough', cols_binarias),
], remainder='drop', verbose_feature_names_out=False)
ct_encoding.set_output(transform='pandas')


# ============================================================================
# 4) PIPELINE: encadena todo en orden. Al hacer fit, cada paso se ajusta solo con
#    los datos de entrenamiento; con transform/predict se aplica igual a test y predecir.
#    En la etapa de modelado se le agrega el estimador al final:
#    Pipeline([('preprocesamiento', preprocesamiento), ('modelo', ...)])
# ============================================================================
preprocesamiento = Pipeline([
    ('eliminar_columnas', FunctionTransformer(eliminar_columnas)),
    ('invalidos_a_nan', FunctionTransformer(invalidos_a_nan)),
    ('recortar_cubierta', RecortarMaximo(columnas=['surface_covered'], percentil=0.999)),
    ('crear_features', FunctionTransformer(crear_features)),
    ('encoding', ct_encoding),
])

# Verificaciones (no ajustan el pipeline: eso se hace en el modelado, solo con train)
assert 'paga_comision' not in X.columns and 'paga_comision' not in X_predecir.columns, "El target se coló en X"
assert list(X.columns) == list(X_predecir.columns), "Entrenamiento y predecir tienen columnas distintas"
assert len(X_predecir) == len(df_casas_predecir), "Se perdieron filas de predecir"
assert X.index.is_unique and X_predecir.index.is_unique, "Hay ids repetidos"
assert len(X) == len(y)

print("X:", X.shape, "| X_predecir:", X_predecir.shape, "| tasa de paga en y:", round(y.mean(), 4))
preprocesamiento  # muestra el diagrama del pipeline

# Función de ganancia y separación train/test

## Función de ganancia y separación train/test

**Por qué:** el enunciado no evalúa con accuracy sino con plata: **+US$ 100** por cada casa que predecimos `paga` y efectivamente paga, y **−US$ 50** por cada casa que predecimos `paga` y no paga. Predecir `no paga` no suma ni resta. Con ~89% de `no paga`, un modelo que nunca dice `paga` tiene 89% de accuracy y ganancia cero.

**Qué hace:** define `ganancia` y la convierte en un *scorer* para que la validación cruzada elija hiperparámetros maximizando plata. Separa un 30% para test (como pide el enunciado), estratificado para que la proporción de `paga` sea la misma en los dos lados. El test se usa solo para informar la ganancia de cada modelo, nunca para elegir nada.

In [ ]:
import math
from sklearn.base import clone
from sklearn.metrics import make_scorer
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict, train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree

def ganancia(y_true, y_pred):
    # y en 0/1 (1 = paga). Solo cuentan las casas que predecimos 'paga'
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    aciertos = ((y_pred == 1) & (y_true == 1)).sum()
    falsos_positivos = ((y_pred == 1) & (y_true == 0)).sum()
    return 100 * aciertos - 50 * falsos_positivos

scorer_ganancia = make_scorer(ganancia)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print(f"Tasa de paga -> train: {y_train.mean():.4f} | test: {y_test.mean():.4f}")
print("Ganancia máxima posible en test (acertar todas las que pagan):", 100 * y_test.sum())

In [ ]:
# Verificaciones de la función de ganancia y del split
assert ganancia(np.array([1, 1, 0]), np.array([1, 0, 0])) == 100   # 1 acierto
assert ganancia(np.array([0, 0]), np.array([1, 1])) == -100         # 2 falsos positivos
assert ganancia(np.array([1, 0]), np.array([1, 1])) == 50           # 100 - 50
assert ganancia(np.array([1, 1]), np.array([0, 0])) == 0            # no predecir paga no cuesta

assert len(X_test) == math.ceil(len(X) * 0.3)                       # train_test_split redondea hacia arriba
assert abs(y_train.mean() - y_test.mean()) < 0.005                  # estratificado
assert X_train.index.intersection(X_test.index).empty               # sin casas en los dos lados

# Modelado: Random Forest
Entrenamos un `RandomForestClassifier` ([documentación](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)) encadenado al **mismo pipeline de preprocesamiento** que el árbol, con la misma separación train/test, la misma validación cruzada y la misma función de ganancia. Así la comparación con el árbol es justa.

**Por qué un bosque:** el árbol solo sobreajusta (con hojas de 10 casas memoriza grupitos) y sus probabilidades toman pocos valores. Un bosque promedia cientos de árboles, cada uno entrenado con una muestra distinta de casas (*bootstrap*) y mirando un subconjunto de variables en cada corte (`max_features`). El promedio reduce la varianza y da probabilidades más finas, lo que además ayuda a elegir el umbral.

`precio_usd` sigue llegando con NaN: `RandomForestClassifier` los maneja solo desde scikit-learn 1.4.

## Ajuste de hiperparámetros

- `max_depth` y `min_samples_leaf`: cuánto puede crecer cada árbol del bosque.
- `max_features`: cuántas variables mira cada corte (`sqrt` ≈ 7 de 48, o el 30% / 50%). Menos variables = árboles más distintos entre sí.
- `class_weight='balanced_subsample'`: más peso a las `paga`, recalculado en cada muestra bootstrap.

**Qué hace:** la grilla completa tiene 150 combinaciones y cada bosque tarda bastante más que un árbol, así que en lugar de `GridSearchCV` usamos `RandomizedSearchCV`, que prueba **20 combinaciones al azar**, con validación cruzada de 5 partes sobre train y puntuando con la ganancia. Durante la búsqueda se fijan 300 árboles: más árboles casi nunca empeoran el bosque, solo lo hacen más lento.

In [ ]:
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

# precio_usd llega con NaN: RandomForestClassifier los acepta desde scikit-learn 1.4
assert tuple(int(p) for p in sklearn.__version__.split('.')[:2]) >= (1, 4), "Se necesita scikit-learn >= 1.4"

pipeline_rf = Pipeline([
    ('preprocesamiento', clone(preprocesamiento)),
    # n_jobs=1 en el bosque: la paralelización la hace la búsqueda (una combinación por núcleo)
    ('modelo', RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=1)),
])
param_dist_rf = {
    'modelo__max_depth': [10, 15, 20, 30, None],
    'modelo__min_samples_leaf': [1, 2, 5, 10, 20],
    'modelo__max_features': ['sqrt', 0.3, 0.5],
    'modelo__class_weight': [None, 'balanced_subsample'],
}
busqueda_rf = RandomizedSearchCV(pipeline_rf, param_dist_rf, n_iter=20, scoring=scorer_ganancia,
                                 cv=cv, n_jobs=-1, random_state=42)
busqueda_rf.fit(X_train, y_train)

resultados_rf = pd.DataFrame(busqueda_rf.cv_results_)
columnas_rf = [f'param_{p}' for p in param_dist_rf] + ['mean_test_score', 'std_test_score']
display(resultados_rf.sort_values('rank_test_score')[columnas_rf].head(10))
print("Mejores hiperparámetros:", busqueda_rf.best_params_)

In [ ]:
modelo_rf = busqueda_rf.best_estimator_   # ya reentrenado con todo X_train
ganancia_rf_train = ganancia(y_train, modelo_rf.predict(X_train))
ganancia_rf_test = ganancia(y_test, modelo_rf.predict(X_test))
print(f"Ganancia train: US$ {ganancia_rf_train:,} | test: US$ {ganancia_rf_test:,}")

assert busqueda_rf.n_splits_ == 5
assert ganancia_rf_test > 0

## Umbral de decisión

Igual que con el árbol: con +100 / −50 conviene decir `paga` cuando la probabilidad supera ~1/3, no 0,5. El umbral se elige con las probabilidades *out-of-fold* de train (`cross_val_predict`); el test no participa.

In [ ]:
proba_oof_rf = cross_val_predict(modelo_rf, X_train, y_train, cv=cv, method='predict_proba', n_jobs=-1)[:, 1]
umbrales_rf = np.arange(0.05, 0.95, 0.01)
ganancias_umbral_rf = [ganancia(y_train, proba_oof_rf >= u) for u in umbrales_rf]
umbral_rf = float(umbrales_rf[np.argmax(ganancias_umbral_rf)])

plt.figure(figsize=(9, 4))
plt.plot(umbrales_rf, ganancias_umbral_rf)
plt.axvline(0.5, color='gray', linestyle='--', label='0,5 (predict)')
plt.axvline(1/3, color='orange', linestyle='--', label='1/3 (teórico)')
plt.axvline(umbral_rf, color='red', label=f'elegido: {umbral_rf:.2f}')
plt.xlabel('umbral de probabilidad para decir "paga"')
plt.ylabel('ganancia en train (validación cruzada)')
plt.title('Random Forest: ganancia según umbral')
plt.legend()
plt.show()

pred_rf = modelo_rf.predict_proba(X_test)[:, 1] >= umbral_rf
ganancia_rf_umbral_test = ganancia(y_test, pred_rf)
vp_rf = int((pred_rf & (y_test == 1)).sum())
fp_rf = int((pred_rf & (y_test == 0)).sum())
fn_rf = int((~pred_rf & (y_test == 1)).sum())
print(f"Umbral elegido: {umbral_rf:.2f}")
print(f"Test: {len(X_test)} casas, {int(y_test.sum())} pagan | VP: {vp_rf} | FP: {fp_rf} | FN: {fn_rf}")
print(pd.Series({
    'Random Forest (umbral 0,5)': ganancia_rf_test,
    f'Random Forest (umbral {umbral_rf:.2f})': ganancia_rf_umbral_test,
    'Máximo posible en test': 100 * int(y_test.sum()),
    'Referencia del enunciado': 166_000,
}, name='Ganancia en test (US$)'))

assert len(proba_oof_rf) == len(X_train)   # el umbral NO se eligió con test
assert vp_rf + fn_rf == y_test.sum()

## ¿Qué variables usa el bosque?

Importancia promedio de cada variable en los 300 árboles (reducción total de impureza; suman 1).

In [ ]:
bosque = modelo_rf[-1]
importancias_rf = pd.Series(bosque.feature_importances_, index=bosque.feature_names_in_).sort_values(ascending=False)
assert abs(importancias_rf.sum() - 1) < 1e-9

plt.figure(figsize=(8, 6))
importancias_rf.head(15).sort_values().plot.barh(color='steelblue')
plt.title('Random Forest: importancia de variables (top 15)')
plt.xlabel('importancia')
plt.show()
print(importancias_rf.head(10).round(3))

## Entregable (Random Forest)

**Qué hace:** reentrena el bosque con **todo** el entrenamiento (train + test) usando los hiperparámetros y el umbral elegidos, predice `casas_predecir` y guarda los ids que predice `paga` en `casas_entrega_random_forest.csv` (archivo aparte, para no pisar el entregable del árbol). Verifica que los ids sean de `casas_predecir` y ninguno de entrenamiento.

In [ ]:
modelo_rf_final = clone(modelo_rf).set_params(modelo__n_jobs=-1).fit(X, y)
proba_predecir_rf = modelo_rf_final.predict_proba(X_predecir)[:, 1]
ids_paga_rf = X_predecir.index[proba_predecir_rf >= umbral_rf]
pd.DataFrame(index=ids_paga_rf).to_csv('casas_entrega_random_forest.csv', index_label='id')

pct_test_rf = pred_rf.mean() * 100
print(f"Casas predichas 'paga': {len(ids_paga_rf)} de {len(X_predecir)} ({len(ids_paga_rf) / len(X_predecir):.1%})")
print(f"En test el modelo predijo 'paga' en el {pct_test_rf:.1f}% de las casas")

assert len(proba_predecir_rf) == len(df_casas_predecir) == 41054
assert ids_paga_rf.isin(df_casas_predecir['id']).all()             # ids de predecir
assert not ids_paga_rf.isin(df_casas_entrenamiento['id']).any()    # ninguno de entrenamiento
assert ids_paga_rf.is_unique and len(ids_paga_rf) > 0
releido_rf = pd.read_csv('casas_entrega_random_forest.csv')
assert list(releido_rf.columns) == ['id'] and len(releido_rf) == len(ids_paga_rf)